# E-commerce Sales Analysis

End-to-end exploratory data analysis of a synthetic Indian e-commerce dataset (2,500+ orders, 2025).
We clean the data, compute business KPIs, and answer key questions about revenue, categories,
regions, payments and discounts.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 2. Load & clean the data

In [ ]:
df = pd.read_csv('../data/ecommerce_sales.csv')
print('Raw shape:', df.shape)
print('Nulls per column:')
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# Remove exact duplicates
df = df.drop_duplicates()
# Parse dates
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
# Fill missing values: median age, mode payment method, 0 discount
df['CustomerAge'] = df['CustomerAge'].fillna(df['CustomerAge'].median())
df['PaymentMethod'] = df['PaymentMethod'].fillna(df['PaymentMethod'].mode()[0])
df['DiscountPct'] = df['DiscountPct'].fillna(0)
print('Clean shape:', df.shape, '| nulls left:', df.isna().sum().sum())

delivered = df[df['OrderStatus'] == 'Delivered'].copy()

## 3. Key business metrics

In [ ]:
total_revenue = delivered['Revenue'].sum()
print(f"Total delivered revenue : Rs.{total_revenue:,.0f}")
print(f"Delivered orders        : {len(delivered)}")
print(f"Unique customers        : {delivered['CustomerID'].nunique()}")
print(f"Average order value     : Rs.{delivered['Revenue'].mean():,.0f}")
print(f"Return rate             : {(df['OrderStatus'] == 'Returned').mean()*100:.1f}%")
print(f"Cancellation rate       : {(df['OrderStatus'] == 'Cancelled').mean()*100:.1f}%")

## 4. Monthly revenue trend — when do we sell most?

In [ ]:
monthly = delivered.set_index('OrderDate').resample('M')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(monthly.index.strftime('%b'), monthly.values, color='#4C78A8')
ax.set_title('Monthly Revenue Trend (2025)', fontweight='bold')
ax.set_ylabel('Revenue (Rs.)')
plt.tight_layout(); plt.show()
# Insight: Oct-Nov festive peak and a July sale spike drive the year.

## 5. Revenue by category — what drives the business?

In [ ]:
cat_rev = delivered.groupby('Category')['Revenue'].sum().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(cat_rev.index, cat_rev.values, color='#72B66A')
ax.set_title('Revenue by Product Category', fontweight='bold')
ax.set_xlabel('Revenue (Rs.)')
plt.tight_layout(); plt.show()
# Insight: Electronics dominates revenue — a concentrated category mix.

## 6. Regional performance

In [ ]:
reg = delivered.groupby('Region')['Revenue'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(reg.index, reg.values, color='#F58518')
ax.set_title('Revenue by Region', fontweight='bold')
ax.set_ylabel('Revenue (Rs.)')
plt.tight_layout(); plt.show()

## 7. How do customers pay?

In [ ]:
pay = delivered['PaymentMethod'].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.pie(pay.values, labels=pay.index, autopct='%1.0f%%', startangle=90,
       pctdistance=0.8, wedgeprops=dict(width=0.4, edgecolor='white'))
ax.set_title('Orders by Payment Method', fontweight='bold')
plt.tight_layout(); plt.show()
# Insight: UPI is the dominant payment mode — checkout must stay UPI-first.

## 8. Do discounts drive revenue?

In [ ]:
delivered['DiscountBucket'] = pd.cut(delivered['DiscountPct'],
    bins=[-1, 0, 10, 20, 100], labels=['No discount', '1-10%', '11-20%', '20%+'])
disc = delivered.groupby('DiscountBucket', observed=True).agg(
    orders=('Revenue', 'size'), revenue=('Revenue', 'sum'))
print(disc.round(0))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(disc.index.astype(str), disc['revenue'].values, color='#B279A2')
ax.set_title('Revenue by Discount Bucket', fontweight='bold')
ax.set_ylabel('Revenue (Rs.)')
plt.tight_layout(); plt.show()
# Insight: most revenue comes from 0-20% discounts; 20%+ discounts add little.

## 9. Top products & customer loyalty

In [ ]:
top_prod = delivered.groupby('ProductName')['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(top_prod.index[::-1], top_prod.values[::-1], color='#54A24B')
ax.set_title('Top 10 Products by Revenue', fontweight='bold')
ax.set_xlabel('Revenue (Rs.)')
plt.tight_layout(); plt.show()

repeat_pct = (delivered['CustomerID'].value_counts() > 1).mean() * 100
print(f'Repeat-customer rate: {repeat_pct:.1f}%')

## 10. Recommendations

1. **Protect the festive peak** — stock up Electronics inventory for Oct-Nov; it drives ~25% of annual revenue.
2. **Rebalance discounts** — deep (>20%) discounts contribute little revenue; cap them and reinvest in the 10-20% band.
3. **UPI-first checkout** — the dominant payment mode; keep it fastest and most reliable.
4. **Grow the long tail** — Electronics is ~78% of revenue; Fashion and Home & Kitchen are the next levers.
5. **Cut cancellations** — a 5% cancel rate is lost revenue; confirm high-value orders faster.